# Nemotron Local CV (Kaggle / RTX PRO 6000)

Runs `nemotron_local_cv.host_metric` against the 600-row stratified val split. Mirrors the host scorer's prompt + extraction + verification byte-for-byte.

Attach: this code dataset, the base model, and the trained adapter (either as another Dataset or as a kernel-source pointing at the training kernel's output).


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Glob-discover instead of hardcoding — Kaggle's path conventions vary
# (`/kaggle/input/datasets/<owner>/<slug>/` for our newer dataset,
#  `/kaggle/input/<slug>/` for older public ones).
import glob, sys
matches = glob.glob("/kaggle/input/**/blackwell_setup.py", recursive=True)
assert matches, "code dataset not attached (no blackwell_setup.py found under /kaggle/input)"
CODE_ROOT = os.path.dirname(matches[0])
print(f"CODE_ROOT = {CODE_ROOT}")
sys.path.insert(0, CODE_ROOT)

import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Pre-model-load Blackwell setup. The mamba_ssm stub MUST be installed before
# AutoModelForCausalLM.from_pretrained loads modeling_nemotron_h.py.
from blackwell_setup import (  # noqa: E402
    set_cuda_alloc_env,
    set_blackwell_ptxas_env,
    install_mamba_ssm_stub,
)
set_cuda_alloc_env()
set_blackwell_ptxas_env()
install_mamba_ssm_stub()
print("CUDA alloc env + Blackwell ptxas env + mamba_ssm stub ready.")


In [ ]:
# peft + vllm for adapter loading/inference. Some Kaggle images include vllm,
# others do not; install from offline packages to keep this kernel deterministic.
# Internet must be OFF for competition compliance, so `--no-index --find-links`
# is mandatory.
import glob, subprocess, sys
matches = glob.glob("/kaggle/input/**/offline_packages", recursive=True)
assert matches, "offline_packages dir not found under /kaggle/input"
OFFLINE_DIR = matches[0]
print(f"OFFLINE_DIR = {OFFLINE_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir",
    "--no-index", "--find-links", OFFLINE_DIR,
    "peft", "vllm",
])
print("peft + vllm installed from offline packages.")

# Compatibility shim for newer protobuf runtimes where MessageFactory.GetPrototype
# was removed (some Kaggle/papermill-side deps still call it).
from google.protobuf import message_factory as _message_factory
if not hasattr(_message_factory.MessageFactory, "GetPrototype"):
    def _get_prototype(self, descriptor):
        return _message_factory.GetMessageClass(descriptor)
    _message_factory.MessageFactory.GetPrototype = _get_prototype
    print("Applied protobuf MessageFactory.GetPrototype compatibility shim.")


In [ ]:
from pathlib import Path

# Use the kernel-metadata.json-attached mount directly (faster than kagglehub.download).
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
assert BASE_MODEL_PATH.is_dir(), f"base model not mounted at {BASE_MODEL_PATH}"
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")
print("  files:", sorted(p.name for p in BASE_MODEL_PATH.iterdir())[:6], "...")

# val_split_600.csv ships inside the code dataset (see scripts/sync_code_to_kaggle.py).
VAL_SPLIT = Path(CODE_ROOT) / "data" / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Run no-training adapter evaluation via the batch CLI.
import sys
from pathlib import Path
import pandas as pd
from nemotron_local_cv import __main__ as cv_cli

SCRIPTS_ROOT = Path(CODE_ROOT) / "scripts"
if SCRIPTS_ROOT.is_dir():
    sys.path.insert(0, str(SCRIPTS_ROOT))
from materialize_merge_experiments import materialize_merge_experiments

manifests = []
static_manifest = Path(CODE_ROOT) / "data" / "adapter_candidates.csv"
if static_manifest.exists():
    manifests.append(static_manifest)

merge_plan = Path(CODE_ROOT) / "data" / "merge_experiments.csv"
if merge_plan.exists():
    merged_root = Path("/kaggle/working/merged_adapters")
    merged_manifest = Path("/kaggle/working/merged_adapter_manifest.csv")
    print(f"merge plan: {merge_plan}")
    materialize_merge_experiments(
        experiments_csv=merge_plan,
        output_root=merged_root,
        manifest_out=merged_manifest,
    )
    manifests.append(merged_manifest)

assert manifests, "missing manifests: expected adapter_candidates.csv and/or merge_experiments.csv"
for m in manifests:
    assert m.exists(), f"missing manifest: {m}"

manifest = Path("/kaggle/working/adapter_manifest_combined.csv")
combined = pd.concat([pd.read_csv(m) for m in manifests], ignore_index=True)
combined = combined.drop_duplicates(subset=["name"], keep="first")
combined.to_csv(manifest, index=False)
print(f"combined {len(combined)} adapters from {len(manifests)} manifest(s)")
print(f"manifest: {manifest}")

summary_out = Path("/kaggle/working/adapter_eval_summary.csv")
debug_dir = Path("/kaggle/working/adapter_debug")
if debug_dir.exists():
    import shutil
    shutil.rmtree(debug_dir)

cmd = [
    "nemotron_local_cv",
    "--adapter-manifest",
    str(manifest),
    "--summary-out",
    str(summary_out),
    "--debug-dir",
    str(debug_dir),
    "--debug",
    "--val",
    str(VAL_SPLIT),
    "--model",
    str(BASE_MODEL_PATH),
]
print("$ " + " ".join(cmd))
sys.argv = cmd
rc = cv_cli.main()
assert rc == 0, f"nemotron_local_cv exited with {rc}"

print(f"wrote {summary_out}")


In [ ]:
import pandas as pd
from pathlib import Path

summary = Path("/kaggle/working/adapter_eval_summary.csv")
assert summary.exists(), "adapter_eval_summary.csv missing"
df = pd.read_csv(summary)
print(df.to_string(index=False))

per_label = Path("/kaggle/working/adapter_eval_summary_per_label.csv")
if per_label.exists():
    pl = pd.read_csv(per_label)
    top = pl.sort_values(["label", "accuracy"], ascending=[True, False]).groupby("label").head(1)
    print("\nTop adapter per label:")
    print(top.to_string(index=False))
